# Inspect observation_spec()

本 notebook 用于在当前环境中打印 `SC2Env.observation_spec()` 以便查看观测空间结构。

说明：如果本地未安装或未配置 StarCraft II / pysc2，创建环境会失败，这个单元包含异常处理以便给出可读的错误信息。

In [ ]:
# 尝试创建一个最小的 SC2Env 并打印 observation_spec()。
# 该单元包含异常处理，若环境不可用会打印错误和回溯。
from absl import flags
# 确保先导入 run_configs 以注册 sc2_run_config 等 flags，再解析它们
import pysc2.run_configs
# 解析 absl flags 并设置 sc2_run_config（Windows 平台使用 'Windows'）
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    # 如果 flags 已解析或其它异常，忽略并继续（后续导入会给出更具体错误）
    pass
try:
    from pysc2.env import sc2_env
    from pysc2.lib import features
    aif = sc2_env.AgentInterfaceFormat(
        feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64),
        use_feature_units=True,
        use_raw_units=True,
        use_unit_counts=True,
        use_camera_position=True,
    )
    env = sc2_env.SC2Env(
        map_name="MoveToBeacon",
        players=[sc2_env.Agent(sc2_env.Race.terran)],
        agent_interface_format=aif,
        step_mul=8,
        game_steps_per_episode=0,
        visualize=False,
    )
    try:
        spec = env.observation_spec()
        # 如果是 dict，逐项打印以便阅读
        if isinstance(spec, dict):
            import pprint
            pp = pprint.PrettyPrinter(width=120)
            pp.pprint(spec)
        else:
            print(spec)
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print("Failed to import or create SC2Env:", e)
    import traceback
    traceback.print_exc()

In [ ]:
# 实验：检查“通用字段”在当前 observation 中是否都出现
from absl import flags
import pysc2.run_configs
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    pass
try:
    from pysc2.env import sc2_env
    aif = sc2_env.AgentInterfaceFormat(
        feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64),
        use_feature_units=True,
        use_raw_units=True,
        use_unit_counts=True,
        use_camera_position=True,
    )
    env = sc2_env.SC2Env(
        map_name="MoveToBeacon",
        players=[sc2_env.Agent(sc2_env.Race.terran)],
        agent_interface_format=aif,
        step_mul=8,
        game_steps_per_episode=0,
        visualize=False,
    )
    try:
        timesteps = env.reset()
        obs = timesteps[0].observation
        common_fields = [
            "action_result",
            "alerts",
            "build_queue",
            "cargo",
            "cargo_slots_available",
            "control_groups",
            "game_loop",
            "last_actions",
            "map_name",
            "multi_select",
            "player",
            "production_queue",
            "score_cumulative",
            "score_by_category",
            "score_by_vital",
            "single_select",
            "upgrades",
            "home_race_requested",
            "away_race_requested",
        ]
        present = [k for k in common_fields if k in obs]
        missing = [k for k in common_fields if k not in obs]
        print("present common fields:", present)
        print("missing common fields:", missing)
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print("Failed to run common-fields check:", e)
    import traceback
    traceback.print_exc()

In [ ]:
# 生成 leader_unit_type / unit_type 的 id -> name 映射（取样输出以验证）
from absl import flags
import pysc2.run_configs
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    pass
try:
    from pysc2.env import sc2_env
    aif = sc2_env.AgentInterfaceFormat(
        feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64),
    )
    env = sc2_env.SC2Env(
        map_name="MoveToBeacon",
        players=[sc2_env.Agent(sc2_env.Race.terran)],
        agent_interface_format=aif,
        step_mul=8,
        game_steps_per_episode=0,
        visualize=False,
    )
    try:
        data = env._controllers[0].data_raw()  # ResponseData
        unit_id_to_name = {u.unit_id: u.name for u in data.units}
        print("unit_id_to_name size:", len(unit_id_to_name))
        # 打印前 20 个（按 id 排序）
        for unit_id in sorted(unit_id_to_name)[:20]:
            print(unit_id, unit_id_to_name[unit_id])
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print("Failed to generate unit_id_to_name mapping:", e)
    import traceback
    traceback.print_exc()

In [ ]:
# 试验：按附录 F 的方法从 ResponseData 中提取 ability_id -> name 映射（不导出 CSV）
from absl import flags
import pysc2.run_configs
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    pass
try:
    from pysc2.env import sc2_env
    aif = sc2_env.AgentInterfaceFormat(
        feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64),
    )
    env = sc2_env.SC2Env(
        map_name="MoveToBeacon",
        players=[sc2_env.Agent(sc2_env.Race.terran)],
        agent_interface_format=aif,
        step_mul=8,
        game_steps_per_episode=0,
        visualize=False,
    )
    try:
        data = env._controllers[0].data_raw()  # ResponseData
        abilities = []
        if hasattr(data, 'abilities'):
            abilities = list(data.abilities)
        elif hasattr(data, 'abilities_data'):
            abilities = list(data.abilities_data)
        else:
            print('ResponseData 属性列表:', [f for f in dir(data) if not f.startswith('_')])
            abilities = []

        ability_id_to_name = {}
        for a in abilities:
            aid = getattr(a, 'ability_id', None)
            name = getattr(a, 'link_name', '') or getattr(a, 'button_name', '')
            if not name:
                try:
                    name = str(a)
                except Exception:
                    name = ''
            if aid is not None:
                ability_id_to_name[aid] = name

        print('ability_id_to_name size:', len(ability_id_to_name))
        # 打印前 40 个样例（按 id 排序）
        for aid in sorted(ability_id_to_name)[:40]:
            print(aid, ability_id_to_name[aid])

        # === 重点：对比同名 ability 的字段差异 ===
        target_name = 'attack'  # 可改为你关心的名字，例如 'move'
        same_name = [a for a in abilities if (getattr(a, 'link_name', '') or getattr(a, 'button_name', '')) == target_name]
        print('\nSame-name abilities for:', target_name, 'count=', len(same_name))
        if len(same_name) == 0:
            print('No abilities matched name:', target_name)
        else:
            # 收集每条 ability 的字段 dict，用于比较差异
            def to_dict(obj):
                try:
                    return {fd.name: val for fd, val in obj.ListFields()}
                except Exception:
                    return {}
            dicts = [to_dict(a) for a in same_name]
            keys = sorted({k for d in dicts for k in d.keys()})
            # 输出每条 ability 的关键字段摘要
            for a, d in zip(same_name, dicts):
                print('ability_id:', getattr(a, 'ability_id', None), 'link_index:', d.get('link_index'), 'button_name:', d.get('button_name'), 'hotkey:', d.get('hotkey'), 'target:', d.get('target'))
            # 逐字段比较是否存在差异
            print('\nField differences:')
            for k in keys:
                values = [d.get(k, None) for d in dicts]
                if len(set(map(str, values))) > 1:
                    print(k, '->', values)
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print('Failed to generate ability mapping:', e)
    import traceback; traceback.print_exc()

In [ ]:
# 调试：打印 abilities 第一个条目的字段以确定正确的属性名
from absl import flags
import pysc2.run_configs
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    pass
try:
    from pysc2.env import sc2_env
    aif = sc2_env.AgentInterfaceFormat(feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64))
    env = sc2_env.SC2Env(map_name="MoveToBeacon", players=[sc2_env.Agent(sc2_env.Race.terran)], agent_interface_format=aif, step_mul=8, game_steps_per_episode=0, visualize=False)
    try:
        data = env._controllers[0].data_raw()
        if hasattr(data, 'abilities'):
            print('abilities count:', len(data.abilities))
            if len(data.abilities) > 0:
                a = data.abilities[0]
                print('ability item type:', type(a))
                try:
                    for fd, val in a.ListFields():
                        print(fd.name, '->', val)
                except Exception:
                    print('dir(a):', [f for f in dir(a) if not f.startswith('_')])
                    print(repr(a))
        else:
            print('ResponseData fields:', [f for f in dir(data) if not f.startswith('_')])
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print('inspect failed:', e)
    import traceback; traceback.print_exc()

In [ ]:
# 列举 s2clientprotocol 中的 ActionResult 枚举（值域与名字）
try:
    from s2clientprotocol import error_pb2 as sc_err
    enum_desc = sc_err.ActionResult.DESCRIPTOR
    print("ActionResult enum values (number -> name):")
    for v in enum_desc.values:
        print(v.number, v.name)
except Exception as e:
    print("Failed to enumerate ActionResult enum:", e)
    import traceback; traceback.print_exc()

In [ ]:
# 列举 s2clientprotocol 中的 Alert 枚举（值域与名字）
try:
    from s2clientprotocol import sc2api_pb2 as sc_pb
    enum_desc = sc_pb.Alert.DESCRIPTOR
    print("Alert enum values (number -> name):")
    for v in enum_desc.values:
        print(v.number, v.name)
except Exception as e:
    print("Failed to enumerate Alert enum:", e)
    import traceback; traceback.print_exc()

In [ ]:
# 验证：从 ResponseData 提取 upgrades 映射的可行性（仅打印示例，不写文件）
from absl import flags
import pysc2.run_configs
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    pass
try:
    from pysc2.env import sc2_env
    aif = sc2_env.AgentInterfaceFormat(feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64))
    env = sc2_env.SC2Env(map_name="MoveToBeacon", players=[sc2_env.Agent(sc2_env.Race.terran)], agent_interface_format=aif, step_mul=8, game_steps_per_episode=0, visualize=False)
    try:
        data = env._controllers[0].data_raw()  # ResponseData
        # 有些版本字段名为 'upgrades' 或 'upgrades_data'，兼容处理
        upgrades = getattr(data, 'upgrades', None) or getattr(data, 'upgrades_data', None) or []
        upgrade_map = {u.upgrade_id: u.name for u in upgrades}
        print('sampled upgrades count:', len(upgrade_map))
        for uid in sorted(upgrade_map)[:50]:
            print(uid, upgrade_map[uid])
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print('Failed to sample upgrades mapping:', e)
    import traceback; traceback.print_exc()

In [ ]:
# Demo：创建带指定请求种族的环境并打印 home/away requested race（仅打印示例，不写文件）
from absl import flags
import pysc2.run_configs
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    pass
try:
    from pysc2.env import sc2_env
    # Race enum is provided in common_pb2 in many s2clientprotocol versions
    try:
        from s2clientprotocol import common_pb2 as sc_common
    except Exception:
        sc_common = None
    # prepare agent interface and players
    aif = sc2_env.AgentInterfaceFormat(feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64))
    players = [sc2_env.Agent(sc2_env.Race.terran, name='agent_terran'), sc2_env.Agent(sc2_env.Race.zerg, name='agent_zerg')]
    env = sc2_env.SC2Env(map_name="Simple64", players=players, agent_interface_format=aif, step_mul=8, game_steps_per_episode=0, visualize=False)
    try:
        timesteps = env.reset()
        obs = timesteps[0].observation
        # Build race_map from common_pb2 if available, otherwise try sc2api_pb2 as a fallback
        race_map = {}
        if sc_common is not None:
            try:
                race_map = {v.number: v.name for v in sc_common.Race.DESCRIPTOR.values}
            except Exception:
                race_map = {}
        if not race_map:
            try:
                from s2clientprotocol import sc2api_pb2 as sc_pb
                race_map = {v.number: v.name for v in sc_pb.Race.DESCRIPTOR.values}
            except Exception:
                race_map = {}
        # helper: robustly extract an integer from observation field
        def _obs_field_to_int(field):
            try:
                # sequence-like (e.g., array or list with one element)
                if hasattr(field, '__len__') and len(field) > 0:
                    return int(field[0])
                return int(field)
            except Exception:
                try:
                    import numpy as _np
                    arr = _np.asarray(field)
                    return int(arr.item())
                except Exception:
                    return None
        if 'home_race_requested' in obs:
            raw = obs['home_race_requested']
            print('home_race_requested raw:', raw)
            h = _obs_field_to_int(raw)
            print('home requested name:', race_map.get(h, 'UNKNOWN'))
        else:
            print('home_race_requested not present in obs')
        if 'away_race_requested' in obs:
            raw = obs['away_race_requested']
            print('away_race_requested raw:', raw)
            a = _obs_field_to_int(raw)
            print('away requested name:', race_map.get(a, 'UNKNOWN'))
        else:
            print('away_race_requested not present in obs')
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print('Demo failed:', e)
    import traceback; traceback.print_exc()


In [ ]:
# Inspect: print contents of the raw response proto (if available)
from absl import flags
import pysc2.run_configs
try:
    flags.FLAGS(['notebook', '--sc2_run_config=Windows'])
except Exception:
    pass
try:
    from pysc2.env import sc2_env
    aif = sc2_env.AgentInterfaceFormat(
        feature_dimensions=sc2_env.Dimensions(screen=64, minimap=64),
        send_observation_proto=True,
    )
    env = sc2_env.SC2Env(map_name="MoveToBeacon", players=[sc2_env.Agent(sc2_env.Race.terran)], agent_interface_format=aif, step_mul=8, game_steps_per_episode=0, visualize=False)
    try:
        timesteps = env.reset()
        obs = timesteps[0].observation
        raw = obs.get('_response_observation')
        print('_response_observation raw type:', type(raw))
        if callable(raw):
            try:
                raw = raw()
            except Exception as e:
                print('Calling raw() failed:', e)
        if raw is None:
            print('_response_observation is None')
        else:
            try:
                print('Proto object type:', type(raw))
                # try ListFields to show which top-level fields exist
                try:
                    fields = list(raw.ListFields())
                    print('ListFields count:', len(fields))
                    for fd, val in fields[:40]:
                        print(fd.name, '->', type(val))
                except Exception as e:
                    print('ListFields failed:', e)
                # show a short sample of attributes
                attrs = [f for f in dir(raw) if not f.startswith('_')]
                print('Sample attrs:', attrs[:60])
            except Exception as e:
                import traceback; traceback.print_exc()
    finally:
        try:
            env.close()
        except Exception:
            pass
except Exception as e:
    print('Inspect failed:', e)
    import traceback; traceback.print_exc()